In [1]:
## Load libraries
import requests
import json
import zipfile
import pandas as pd
import numpy as np
import geopandas as gpd
import matplotlib.pyplot as plt
import contextily as ctx
from concurrent.futures import ThreadPoolExecutor, as_completed

## Input contract

This notebook assumes as input a **GeoDataFrame of points** with the following properties:

- Each row represents a **hexagon centroid**
- A unique and non-null `point_id` column exists
- The active geometry is of type **Point**
- The CRS matches the **project canonical CRS**

⚠️ **If any of these conditions are not met, the analysis is not valid.**

In [8]:
## Global parameters
PROJECT_CRS = "EPSG:4326"  # Project canonical CRS
METRIC_CRS = "EPSG:3763"   # CRS for planar projection
RADIO_WALK = 4000          # # Geometric radius-based pre-filter for points
OTP_BASE_URL = "http://otp-serve:8080/otp/routers/default/plan"
WALK_DATE = "12-10-2025"
WALK_TIME = "8:00am"

In [9]:
points = gpd.read_file("/results/points_hex_500m_wgs84.geojson")
zip_path_mdp = "/data/mdp_gtfs.zip"

In [10]:
assert isinstance(points, gpd.GeoDataFrame), "Input is not a GeoDataFrame"
assert points.geom_type.unique().tolist() == ["Point"], "Geometry is not Point"
assert points.crs == PROJECT_CRS, "Incorrect CRS in points"
assert "point_id" in points.columns, "Missing point_id column"
assert points["point_id"].is_unique, "point_id is not unique"
assert points["point_id"].isna().sum() == 0, "point_id contains null values"